# India Census 2011 — Visualization Suite
**640 Districts · 35 States/UTs · 118 Metrics**

This is the master index for the India Census 2011 deep-dive visualization suite.  
Each notebook is self-contained and reads directly from `data/processed/census_model.db`.

---

| # | Notebook | Topics Covered | Charts |
|---|---|---|---|
| 01 | [Demographics & Population](01_demographics.ipynb) | Population, sex ratio, age groups, SC/ST | Treemap, Bars, Scatter, Histogram |
| 02 | [Literacy & Education](02_literacy_education.ipynb) | Literacy rates, gender gap, education funnel | Grouped bars, Funnel, Stacked, Box plot |
| 03 | [Urbanization](03_urbanization.ipynb) | Urban/rural split, electricity, internet, LPG | Bars, Heatmap, Scatter, Histogram |
| 04 | [Economic Activity](04_economic_activity.ipynb) | Worker types, dependency ratio, income parity | Grouped bars, Stacked, Pie, Heatmap, Scatter |
| 05 | [Religion & Social Demographics](05_religion_social.ipynb) | Religious composition, diversity index | Pie, Stacked, Heatmap |
| 06 | [Household Amenities & Dev Index](06_household_amenities.ipynb) | Electricity, LPG, sanitation, composite index | Heatmap, Scatter, Histogram, Bars |
| 07 | [Assets & Lifestyle](07_assets_lifestyle.ipynb) | TV, mobile, bicycle, car, scooter ownership | Bars, Heatmap, Scatter, Radar |
| 08 | [Housing & Infrastructure](08_housing_infrastructure.ipynb) | Water sources, dilapidated housing, HH sizes | Stacked, Bars, Heatmap |

---

## Quick-start

In [ ]:
import sqlite3
import pandas as pd

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name, f.*
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

print(f'Shape: {df.shape}')
print(f'States/UTs: {df.state_name.nunique()}')
print(f'Districts: {df.district_name.nunique()}')
print(f'Total Population: {df.population.sum():,.0f}')
print(f'Total Households: {df.households.sum():,.0f}')
print()
print('Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:3}. {col}')

## State-level summary

In [ ]:
import plotly.express as px

summary = df.groupby('state_name').agg(
    districts=('district_name','count'),
    population=('population','sum'),
    literacy_rate=('literate', lambda x: (x.sum() / df.loc[x.index, 'population'].sum() * 100).round(1)),
    households=('households','sum'),
).reset_index().sort_values('population', ascending=False)

fig = px.bar(
    summary,
    x='state_name', y='population',
    color='literacy_rate',
    color_continuous_scale='RdYlGn',
    hover_data={'districts': True, 'households': ':,'},
    title='Population by State — coloured by Literacy Rate',
    labels={'population': 'Population', 'state_name': 'State', 'literacy_rate': 'Literacy %'}
)
fig.update_layout(xaxis_tickangle=-45, height=500, coloraxis_colorbar_title='Literacy %')
fig.show()

print(summary.to_string(index=False))

---

## Key National Statistics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

stats = {
    'Total Population':          f"{df['population'].sum():,.0f}",
    'Total Households':          f"{df['households'].sum():,.0f}",
    'Literacy Rate':             f"{df['literate'].sum()/df['population'].sum()*100:.2f}%",
    'Female Literacy Rate':      f"{df['female_literate'].sum()/df['female'].sum()*100:.2f}%",
    'Sex Ratio':                 f"{df['female'].sum()/df['male'].sum()*1000:.0f} per 1000 males",
    'Urban HH %':                f"{df['urban_households'].sum()/df['households'].sum()*100:.2f}%",
    'Electricity Access':        f"{df['housholds_with_electric_lighting'].sum()/df['households'].sum()*100:.2f}%",
    'LPG/PNG Access':            f"{df['lpg_or_png_households'].sum()/df['households'].sum()*100:.2f}%",
    'Internet Access':           f"{df['households_with_internet'].sum()/df['households'].sum()*100:.2f}%",
    'Mobile Phone Ownership':    f"{df['households_with_telephone_mobile_phone'].sum()/df['households'].sum()*100:.2f}%",
    'TV Ownership':              f"{df['households_with_television'].sum()/df['households'].sum()*100:.2f}%",
    'Open Defecation Rate':      f"{df['not_having_latrine_facility_within_the_premises_alternative_source_open_households'].sum()/df['households'].sum()*100:.2f}%",
}

fig, ax = plt.subplots(figsize=(9, 6))
ax.axis('off')
rows = [[k, v] for k, v in stats.items()]
table = ax.table(cellText=rows, colLabels=['Metric', 'Value'],
                 cellLoc='left', loc='center',
                 colWidths=[0.65, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1565c0')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#e3f2fd')
    cell.set_edgecolor('#ccc')

ax.set_title('India Census 2011 — Key National Statistics', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()